# BovDB: Top and Bottom Analysis on B3 Stock Quotes

This notebook demonstrates how to connect to the **BovDB SQLite database**, extract historical stock quotes for Brazilian equities, apply corporate adjustment factors, and compute local extrema (tops and bottoms) for algorithmic analysis.

## 1. Setup and Imports

In [ ]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.signal import find_peaks

# Add src folder if running from within the repository
if os.path.exists('../src'):
    sys.path.insert(0, '../src')

print("Libraries loaded successfully.")

## 2. Database Connection Helper

In [ ]:
def get_db_connection(db_path=None):
    if db_path is None:
        # Check common locations
        candidates = [
            '../DataBase/DataBase.db',
            '../DataBase/Database_define.db',
            'DataBase/DataBase.db',
            'DataBase/Database_define.db'
        ]
        for c in candidates:
            if os.path.exists(c):
                db_path = c
                break
    if db_path is None or not os.path.exists(db_path):
        print(f"Note: Database file not found at default paths. Using path: {db_path}")
    return sqlite3.connect(db_path if db_path else 'DataBase/DataBase.db')

## 3. Querying Historical Daily Data

In [ ]:
def load_stock_data(conn, ticker_symbol='PETR4', start_date='2018-01-01', end_date='2024-06-30'):
    query = f"""
    SELECT 
        p.date,
        p.open,
        p.high,
        p.low,
        p.close,
        p.volume,
        p.factor
    FROM price p
    JOIN ticker t ON p.id_ticker = t.id_ticker
    WHERE t.cd_ticker = '{ticker_symbol}'
      AND p.date BETWEEN '{start_date}' AND '{end_date}'
    ORDER BY p.date ASC
    """
    df = pd.read_sql_query(query, conn)
    if not df.empty:
        df['date'] = pd.to_datetime(df['date'])
        # Compute adjusted prices
        df['close_adj'] = df['close'] / df['factor']
        df['open_adj'] = df['open'] / df['factor']
        df['high_adj'] = df['high'] / df['factor']
        df['low_adj'] = df['low'] / df['factor']
    return df

## 4. Peak and Valley Detection (Tops and Bottoms)

In [ ]:
def detect_extrema(df, distance=20, prominence=1.0):
    """
    Detects local peaks (tops) and valleys (bottoms) using scipy find_peaks.
    """
    prices = df['close_adj'].values
    peaks, _ = find_peaks(prices, distance=distance, prominence=prominence)
    troughs, _ = find_peaks(-prices, distance=distance, prominence=prominence)
    
    df['is_peak'] = False
    df['is_trough'] = False
    df.loc[peaks, 'is_peak'] = True
    df.loc[troughs, 'is_trough'] = True
    return df, peaks, troughs

## 5. Interactive Chart with Extrema Markers

In [ ]:
def plot_extrema_chart(df, peaks, troughs, ticker_symbol='PETR4'):
    fig = go.Figure()
    
    # Base adjusted close price line
    fig.add_trace(go.Scatter(
        x=df['date'],
        y=df['close_adj'],
        mode='lines',
        name=f'{ticker_symbol} Adjusted Close',
        line=dict(color='#1f77b4', width=1.5)
    ))
    
    # Peaks (Tops)
    fig.add_trace(go.Scatter(
        x=df['date'].iloc[peaks],
        y=df['close_adj'].iloc[peaks],
        mode='markers',
        name='Local Peaks (Tops)',
        marker=dict(symbol='triangle-up', size=10, color='red')
    ))
    
    # Troughs (Bottoms)
    fig.add_trace(go.Scatter(
        x=df['date'].iloc[troughs],
        y=df['close_adj'].iloc[troughs],
        mode='markers',
        name='Local Valleys (Bottoms)',
        marker=dict(symbol='triangle-down', size=10, color='green')
    ))
    
    fig.update_layout(
        title=f'{ticker_symbol} - Top and Bottom Extrema Detection',
        xaxis_title='Date',
        yaxis_title='Adjusted Price (R$)',
        template='plotly_white',
        hovermode='x unified'
    )
    
    return fig

# Example invocation when database is available:
# conn = get_db_connection()
# df = load_stock_data(conn, 'PETR4')
# if not df.empty:
#     df, peaks, troughs = detect_extrema(df)
#     fig = plot_extrema_chart(df, peaks, troughs, 'PETR4')
#     fig.show()
# conn.close()